# Aligning MERFISH to Visium H&E with point annotations (squidpy + STalign)

In this notebook, we align MERFISH cell positions to a Visium H&E staining image using point-based landmark annotations.

Since this is a point-to-image alignment, we use the STalign/LDDMM backend via `squidpy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load MERFISH data (source)

In [ ]:
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)

coords_source = np.column_stack([df1['center_x'].values, df1['center_y'].values])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)), obs=df1)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2)
ax.set_title('MERFISH cell positions (source)')

## Load H&E image (target)

In [ ]:
image_file = '../visium_data/tissue_hires_image.png'
V = plt.imread(image_file)

fig, ax = plt.subplots()
ax.imshow(V)
ax.set_title('H&E staining image (target)')

## Load landmark points

In [ ]:
# Load point-based landmarks
pointsIlist = np.load('../visium_data/Merfish_S2_R3_points.npy', allow_pickle=True).tolist()
pointsJlist = np.load('../visium_data/tissue_hires_image_points.npy', allow_pickle=True).tolist()

pointsI = []
pointsJ = []
for i in pointsIlist.keys():
    for j in range(len(pointsIlist[i])):
        pointsI.append([pointsIlist[i][j][1], pointsIlist[i][j][0]])
for i in pointsJlist.keys():
    for j in range(len(pointsJlist[i])):
        pointsJ.append([pointsJlist[i][j][1], pointsJlist[i][j][0]])

pointsI = np.array(pointsI)
pointsJ = np.array(pointsJ)
print(f"{len(pointsI)} source landmarks, {len(pointsJ)} target landmarks")

## Align using STalign (point-to-image)

In [ ]:
sq.experimental.tl.align(
    adata_source,
    V,
    method='lddmm',
    resolution=30.0,
    niter=200,
    diffeo_start=100,
    landmark_source=pointsI,
    landmark_target=pointsJ,
    verbose=True,
    sigmaP=2e-1,
    sigmaM=0.18,
    sigmaB=0.18,
    sigmaA=0.18,
    epL=5e-11,
    epT=5e-4,
    epV=5e1,
)

## Visualize results

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.imshow(V)
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='MERFISH aligned')
ax.scatter(pointsJ[:, 1], pointsJ[:, 0], c='red', label='target landmarks', s=100)
ax.set_title('Aligned MERFISH on H&E')
ax.legend(markerscale=5)

In [ ]:
df_aligned = pd.DataFrame({'aligned_x': aligned[:, 0], 'aligned_y': aligned[:, 1]})
results = pd.concat([df1, df_aligned], axis=1)
results.head()